# CycleGAN — Enhanced: SW-MSA Generator + Frequency-Aware Discriminator + SSIM Loss

**Three novelties added over baseline:**
1. **True SW-MSA** — alternating W-MSA/SW-MSA with cyclic shift (cross-window context, the defining Swin property)
2. **Frequency-Aware PatchGAN** — FFT log-magnitude branch fused with spatial branch (penalises spectral artefacts)
3. **SSIM Loss** — 1−SSIM added to generator objective (training aligned with evaluation metric)

*All other cells (data loader, visualisation, saving) are identical to the baseline.*

In [ ]:
# Cell 0 — Imports
import os, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
import pandas as pd
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
print('TF:', tf.__version__)

In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Config & Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ------------------------------------------------------------------
# Change these paths to the local dataset location.
# MRI data are NOT included in the GitHub repository.
# ------------------------------------------------------------------
DATA_ROOT = "/path/to/CycleGan"
T1_DIR = os.path.join(DATA_ROOT, "T1 healthy images", "T1 healthy images")
T2_DIR = os.path.join(DATA_ROOT, "T2 Healthy images", "T2 Healthy images")

IMG_HEIGHT, IMG_WIDTH = 256, 256
BATCH_SIZE = 2
EPOCHS = 100
OUTPUT_DIR = os.path.join(DATA_ROOT, "CycleGAN_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Loss weights
LAMBDA      = 10
LAMBDA_ID   = 0.5
LAMBDA_PERC = 0.5
LAMBDA_SSIM = 0.3

print('Config ready.')


In [ ]:
# Cell 3 — Instance Normalisation (unchanged from baseline)
class InstanceNormalization(layers.Layer):
    def __init__(self, epsilon=1e-5):
        super().__init__()
        self.epsilon = epsilon

    def build(self, input_shape):
        self.gamma = self.add_weight(shape=(input_shape[-1],), initializer='ones',  trainable=True)
        self.beta  = self.add_weight(shape=(input_shape[-1],), initializer='zeros', trainable=True)

    def call(self, x):
        mean, var = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        return self.gamma * (x - mean) / tf.sqrt(var + self.epsilon) + self.beta

In [ ]:
# Cell 4 — VGG19 Perceptual Loss (unchanged from baseline)
def build_vgg_model():
    vgg = VGG19(include_top=False, weights='imagenet', input_shape=(256, 256, 3))
    vgg.trainable = False
    return tf.keras.Model(vgg.input, vgg.get_layer('block3_conv3').output)

vgg_model = build_vgg_model()

def perceptual_loss(y_true, y_pred):
    y_true = (y_true + 1.0) * 127.5
    y_pred = (y_pred + 1.0) * 127.5
    y_true_rgb = tf.image.grayscale_to_rgb(y_true)
    y_pred_rgb = tf.image.grayscale_to_rgb(y_pred)
    real_feat = vgg_model(preprocess_input(y_true_rgb))
    fake_feat = vgg_model(preprocess_input(y_pred_rgb))
    return tf.reduce_mean(tf.abs(real_feat - fake_feat))

print('Perceptual loss ready.')

In [ ]:
# Cell 5 — SSIM Loss  [NOVELTY]
#
# Minimising 1-SSIM directly maximises structural similarity during training.
# This aligns the training objective with the evaluation metric (Table 2)
# and preserves fine MRI structural details — sulcal boundaries, ventricular
# morphology, iron deposit patterns — beyond what pixel-wise L1 alone achieves.
#
def ssim_loss(y_true, y_pred):
    """Returns 1 - SSIM so minimising this = maximising structural similarity."""
    yt = tf.clip_by_value((y_true + 1.0) * 127.5, 0.0, 255.0)
    yp = tf.clip_by_value((y_pred + 1.0) * 127.5, 0.0, 255.0)
    yt_rgb = tf.image.grayscale_to_rgb(yt)
    yp_rgb = tf.image.grayscale_to_rgb(yp)
    return 1.0 - tf.reduce_mean(tf.image.ssim(yt_rgb, yp_rgb, max_val=255.0))

print('SSIM loss ready.')

In [ ]:
# Cell 6 — Window Attention (base of Swin blocks; unchanged from baseline)
class WindowAttention(layers.Layer):
    """
    Local-window multi-head self-attention.
    Feature maps are split into non-overlapping windows; attention is
    computed independently within each window, giving O(n) complexity
    in the number of windows.
    """
    def __init__(self, dim, window_size=8, num_heads=4):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        assert dim % num_heads == 0
        self.head_dim = dim // num_heads
        self.scale    = self.head_dim ** -0.5

    def build(self, input_shape):
        self.qkv  = layers.Dense(self.dim * 3)
        self.proj = layers.Dense(self.dim)

    def call(self, x):
        B = tf.shape(x)[0]; H = tf.shape(x)[1]
        W = tf.shape(x)[2]; C = tf.shape(x)[3]
        ws = self.window_size
        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws
        x = tf.pad(x, [[0,0],[0,pad_h],[0,pad_w],[0,0]])
        Hp = tf.shape(x)[1]; Wp = tf.shape(x)[2]
        x_win = tf.image.extract_patches(
            images=x, sizes=[1,ws,ws,1], strides=[1,ws,ws,1],
            rates=[1,1,1,1], padding='VALID')
        x_win = tf.reshape(x_win, [-1, ws*ws, C])
        qkv = self.qkv(x_win)
        qkv = tf.reshape(qkv, [-1, ws*ws, 3, self.num_heads, self.head_dim])
        qkv = tf.transpose(qkv, perm=[2,0,3,1,4])
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = tf.nn.softmax(
            tf.matmul(q, k, transpose_b=True) * self.scale, axis=-1)
        out  = tf.matmul(attn, v)
        out  = tf.transpose(out, perm=[0,2,1,3])
        out  = tf.reshape(out, [-1, ws*ws, C])
        out  = self.proj(out)
        out  = tf.reshape(out, [-1, Hp//ws, Wp//ws, ws, ws, C])
        out  = tf.transpose(out, [0,1,3,2,4,5])
        out  = tf.reshape(out, [-1, Hp, Wp, C])
        return out[:, :H, :W, :]

In [ ]:
# Cell 7 — True Shifted-Window Swin Block (SW-MSA)  [NOVELTY — CORE FIX]
#
# WHAT WAS WRONG IN BASELINE:
#   def SwinBlock(x, dim, num_heads, window_size=8, shift_size=0):
#       ...  # shift_size parameter existed but was NEVER USED
#       x = WindowAttention(dim, window_size, num_heads)(x)  # always W-MSA
#
# WHY IT MATTERS:
#   Without the cyclic shift, every block is standard W-MSA. Windows cannot
#   exchange information across their fixed boundaries. Cross-window context
#   is the defining architectural property of Swin Transformers and is what
#   justifies calling the architecture 'Swin-UNet' in the paper.
#
# WHAT IS FIXED HERE:
#   - layer_idx tracked globally through the entire network
#   - Even-indexed layers => W-MSA  (no shift)
#   - Odd-indexed  layers => SW-MSA (cyclic shift of window_size//2)
#   - Cyclic roll applied BEFORE attention, reversed AFTER attention
#
def SwinBlock(x, dim, num_heads, window_size=8, layer_idx=0):
    """
    layer_idx: global index in the network.
      even => W-MSA  (standard window attention, no shift)
      odd  => SW-MSA (cyclic-shifted window attention, cross-window context)
    """
    shift_size = (window_size // 2) if (layer_idx % 2 == 1) else 0

    shortcut = x
    x = layers.LayerNormalization(epsilon=1e-5)(x)

    # --- Apply cyclic shift before attention (SW-MSA path only) ---
    if shift_size > 0:
        x = tf.roll(x, shift=[-shift_size, -shift_size], axis=[1, 2])

    x = WindowAttention(dim, window_size, num_heads)(x)

    # --- Reverse cyclic shift after attention ---
    if shift_size > 0:
        x = tf.roll(x, shift=[shift_size, shift_size], axis=[1, 2])

    x = layers.Add()([shortcut, x])        # residual 1

    shortcut2 = x
    x = layers.LayerNormalization(epsilon=1e-5)(x)
    x = layers.Dense(dim * 4, activation='gelu')(x)
    x = layers.Dense(dim)(x)
    x = layers.Add()([shortcut2, x])       # residual 2
    return x

print('SW-MSA SwinBlock ready.')

In [ ]:
# Cell 8 — Generator: Swin-UNet with proper SW-MSA alternation  [NOVELTY]
#
# layer_idx is incremented globally so the W/SW alternation is
# consistent across encoder (0-3), bottleneck (4-5), and decoder (6-10).
#
def build_generator():
    inputs    = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1))
    layer_idx = 0   # global W/SW counter

    # --- Stem ---
    x = layers.Conv2D(64, 3, strides=2, padding='same')(inputs)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)
    x = SwinBlock(x, 64, num_heads=4, layer_idx=layer_idx); layer_idx += 1   # W-MSA

    # --- Encoder ---
    enc_filters = [128, 256, 512]
    skips = [x]
    for f in enc_filters:
        x = layers.Conv2D(f, 3, strides=2, padding='same')(x)
        x = InstanceNormalization()(x)
        x = layers.ReLU()(x)
        x = SwinBlock(x, f, num_heads=4, layer_idx=layer_idx); layer_idx += 1
        skips.append(x)

    # --- Bottleneck: two alternating Swin blocks for richer context ---
    x = layers.Conv2D(512, 3, padding='same')(x)
    x = InstanceNormalization()(x)
    x = SwinBlock(x, 512, num_heads=4, layer_idx=layer_idx); layer_idx += 1  # W-MSA
    x = SwinBlock(x, 512, num_heads=4, layer_idx=layer_idx); layer_idx += 1  # SW-MSA

    # --- Decoder with skip connections ---
    dec_filters = [512, 256, 128, 64]
    for f, skip in zip(dec_filters, reversed(skips)):
        if f != 512:
            x = layers.Conv2DTranspose(f, 3, strides=2, padding='same')(x)
            x = InstanceNormalization()(x)
            x = layers.ReLU()(x)
        x = SwinBlock(x, f, num_heads=4, layer_idx=layer_idx); layer_idx += 1
        x = layers.Concatenate()([x, skip])

    # --- Final upsample ---
    x = layers.Conv2DTranspose(64, 3, strides=2, padding='same')(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)
    x = SwinBlock(x, 64, num_heads=4, layer_idx=layer_idx)

    # --- Output head ---
    x = layers.Conv2D(1, 1, padding='same', activation='tanh')(x)
    return Model(inputs, x, name='Generator_SWMSA')

print('Generator defined.')

In [ ]:
# Cell 9 — Frequency-Aware PatchGAN Discriminator  [NOVELTY]
#
# MOTIVATION:
#   PD diagnosis depends critically on iron-sensitive HIGH-FREQUENCY signals
#   in the substantia nigra visible on T2-weighted MRI. Standard PatchGAN
#   operates only in the spatial domain and cannot penalise spectral artefacts.
#   Generated images may look spatially plausible but lack the iron-sensitive
#   frequency content that is essential for PD characterisation.
#
# DESIGN — Two parallel branches, fused before the patch-score output:
#
#   Branch A (Spatial PatchGAN):
#     Conv(64,4,s2) -> LReLU
#     Conv(128,4,s2) -> IN -> LReLU
#     Conv(256,4,s2) -> IN -> LReLU
#     Conv(512,4,s2) -> IN -> LReLU
#     Output shape: (B, H/16, W/16, 512)
#
#   Branch B (Frequency FFT log-magnitude):
#     FFT2D(input) -> DC-shift -> log(|·| + 1e-8)
#     Conv(64,3,s2)  -> LReLU
#     Conv(128,3,s2) -> LReLU
#     Conv(256,3,s2) -> LReLU
#     Conv(512,3,s2) -> LReLU
#     Output shape: (B, H/16, W/16, 512)  -- matches Branch A
#
#   Fusion:
#     Concat(A, B) -> (B, H/16, W/16, 1024)
#     Conv(512,4,s1) -> IN -> LReLU
#     Conv(1, 4) -> patch scores
#
def build_discriminator():
    inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1), name='disc_input')

    # === Branch A: Spatial PatchGAN ===
    s = layers.Conv2D(64, 4, strides=2, padding='same')(inputs)
    s = layers.LeakyReLU(0.2)(s)
    for filt in [128, 256, 512]:
        s = layers.Conv2D(filt, 4, strides=2, padding='same')(s)
        s = InstanceNormalization()(s)
        s = layers.LeakyReLU(0.2)(s)
    # s shape: (B, H/16, W/16, 512)

    # === Branch B: Frequency (FFT log-magnitude) ===
    def freq_branch(inp):
        """2D FFT -> DC-centred shift -> log-magnitude. Returns (B,H,W,1)."""
        xc  = tf.cast(inp[..., 0], tf.complex64)      # (B, H, W)
        fft = tf.signal.fftshift(tf.signal.fft2d(xc)) # centre DC component
        mag = tf.math.log(tf.abs(fft) + 1e-8)         # log magnitude
        return tf.expand_dims(mag, axis=-1)            # (B, H, W, 1)

    f = layers.Lambda(freq_branch, name='fft_log_mag')(inputs)
    for filt in [64, 128, 256, 512]:
        f = layers.Conv2D(filt, 3, strides=2, padding='same')(f)
        f = layers.LeakyReLU(0.2)(f)
    # f shape: (B, H/16, W/16, 512) -- matches Branch A

    # === Fusion ===
    x = layers.Concatenate()([s, f])                   # (B, H/16, W/16, 1024)
    x = layers.Conv2D(512, 4, strides=1, padding='same')(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2D(1, 4, padding='same')(x)         # patch-score map
    return Model(inputs, x, name='FreqAware_PatchGAN')

print('Frequency-aware discriminator defined.')

In [ ]:
# Cell 10 — Data Loader with Subject-Level Pairing
#
# The loader preserves the original slice pairing while also extracting
# subject IDs. Subject IDs are later used for train/validation splitting
# so that no subject contributes slices to both partitions.
#
# Expected filename convention:
# SUBJECT_PART_1_SUBJECT_PART_2_SUBJECT_PART_3_sliceXX.ext
# If your filenames use another convention, edit extract_subject_id().

import re
from pathlib import Path

def extract_subject_id(filename):
    stem = Path(filename).stem
    parts = stem.split('_')

    for i, part in enumerate(parts):
        if re.fullmatch(r'slice[-_]?\d+', part, flags=re.IGNORECASE):
            subject = '_'.join(parts[:i])
            if subject:
                return subject

    m = re.match(r'^(.*?)[_-]s\d+$', stem, flags=re.IGNORECASE)
    if m:
        return m.group(1)

    return stem

def extract_slice_key(filename):
    """
    Key used to match the corresponding T1 and T2 slice.
    """
    stem = Path(filename).stem
    parts = stem.split('_')

    for i, part in enumerate(parts):
        if re.fullmatch(r'slice[-_]?\d+', part, flags=re.IGNORECASE):
            return '_'.join(parts[:i+1])

    m = re.search(r'([_-]s\d+)$', stem, flags=re.IGNORECASE)
    if m:
        return stem[:m.start()] + m.group(1)

    return stem

def build_domain_manifest(folder):
    records = []

    valid_exts = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}

    for fname in sorted(os.listdir(folder)):
        path = os.path.join(folder, fname)
        if not os.path.isfile(path):
            continue
        if Path(fname).suffix.lower() not in valid_exts:
            continue

        records.append({
            'filename': fname,
            'filepath': path,
            'subject_id': extract_subject_id(fname),
            'slice_key': extract_slice_key(fname)
        })

    return pd.DataFrame(records)

t1_manifest = build_domain_manifest(T1_DIR)
t2_manifest = build_domain_manifest(T2_DIR)

# Match the corresponding T1/T2 slices.
t1_map = {
    row['slice_key']: row['filepath']
    for _, row in t1_manifest.iterrows()
}
t2_map = {
    row['slice_key']: row['filepath']
    for _, row in t2_manifest.iterrows()
}

common_keys = sorted(set(t1_map) & set(t2_map))

pairs = []
for key in common_keys:
    subject_id = extract_subject_id(Path(t1_map[key]).name)
    pairs.append({
        'slice_key': key,
        'subject_id': subject_id,
        't1_path': t1_map[key],
        't2_path': t2_map[key]
    })

pairs_df = pd.DataFrame(pairs)

if pairs_df.empty:
    raise RuntimeError(
        'No T1/T2 pairs were found. Check T1_DIR, T2_DIR and the '
        'filename parsing functions.'
    )

print(f'Matched T1/T2 slices: {len(pairs_df)}')
print(f'Matched subjects: {pairs_df.subject_id.nunique()}')

# Sanity check: a subject must not map to conflicting IDs.
print('Example matched pair:')
print(pairs_df.head(1).to_string(index=False))


In [ ]:
# Cell 11 — SSIM/PSNR helpers + visualisation (unchanged from baseline)
def evaluate_ssim_psnr(real, generated):
    real      = (real      + 1.0) * 127.5
    generated = (generated + 1.0) * 127.5
    real = real.squeeze(); generated = generated.squeeze()
    dr = real.max() - real.min()
    return (ssim(real, generated, data_range=dr),
            psnr(real, generated, data_range=dr))

def save_image_grid(t1, fake_t2, real_t2, filename):
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    for ax, img, title in zip(axs, [t1, fake_t2, real_t2],
                               ['T1 Input', 'Generated T2', 'Real T2']):
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(title); ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename))
    plt.close()

In [ ]:
# Cell 12 — Training Setup
# Total generator G loss:
#   L = L_adv + LAMBDA*L_cyc + LAMBDA_ID*LAMBDA*L_id
#     + LAMBDA_PERC*L_perc + LAMBDA_SSIM*L_ssim   [SSIM term is NOVELTY]
#
# The frequency-aware discriminator automatically provides richer
# spectral adversarial gradients without any extra loss term.
#
loss_obj     = tf.keras.losses.MeanSquaredError()
optimizer_g  = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
optimizer_f  = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
optimizer_dx = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
optimizer_dy = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

@tf.function
def train_step(real_x, real_y, generator_g, generator_f,
               discriminator_x, discriminator_y):
    with tf.GradientTape(persistent=True) as tape:
        fake_y   = generator_g(real_x, training=True)
        fake_x   = generator_f(real_y, training=True)
        cycled_x = generator_f(fake_y, training=True)
        cycled_y = generator_g(fake_x, training=True)
        same_x   = generator_f(real_x, training=True)
        same_y   = generator_g(real_y, training=True)

        disc_real_x = discriminator_x(real_x, training=True)
        disc_real_y = discriminator_y(real_y, training=True)
        disc_fake_x = discriminator_x(fake_x, training=True)
        disc_fake_y = discriminator_y(fake_y, training=True)

        # Adversarial
        gen_g_loss = loss_obj(tf.ones_like(disc_fake_y), disc_fake_y)
        gen_f_loss = loss_obj(tf.ones_like(disc_fake_x), disc_fake_x)

        # Cycle-consistency
        cyc_loss = (tf.reduce_mean(tf.abs(real_x - cycled_x)) +
                    tf.reduce_mean(tf.abs(real_y - cycled_y)))

        # Identity
        id_loss  = (tf.reduce_mean(tf.abs(real_x - same_x)) +
                    tf.reduce_mean(tf.abs(real_y - same_y)))

        # Perceptual (VGG19)
        perc_loss = perceptual_loss(real_y, fake_y)

        # SSIM loss  [NOVELTY]
        ssim_l = ssim_loss(real_y, fake_y)

        # Total generator losses
        total_gen_g = (gen_g_loss
                       + LAMBDA      * cyc_loss
                       + LAMBDA_ID   * LAMBDA * id_loss
                       + LAMBDA_PERC * perc_loss
                       + LAMBDA_SSIM * ssim_l)      # <-- SSIM term added
        total_gen_f = (gen_f_loss
                       + LAMBDA    * cyc_loss
                       + LAMBDA_ID * LAMBDA * id_loss)

        # Discriminator
        disc_x_loss = (loss_obj(tf.ones_like(disc_real_x),  disc_real_x) +
                       loss_obj(tf.zeros_like(disc_fake_x), disc_fake_x))
        disc_y_loss = (loss_obj(tf.ones_like(disc_real_y),  disc_real_y) +
                       loss_obj(tf.zeros_like(disc_fake_y), disc_fake_y))

    optimizer_g.apply_gradients(
        zip(tape.gradient(total_gen_g, generator_g.trainable_variables),
            generator_g.trainable_variables))
    optimizer_f.apply_gradients(
        zip(tape.gradient(total_gen_f, generator_f.trainable_variables),
            generator_f.trainable_variables))
    optimizer_dx.apply_gradients(
        zip(tape.gradient(disc_x_loss, discriminator_x.trainable_variables),
            discriminator_x.trainable_variables))
    optimizer_dy.apply_gradients(
        zip(tape.gradient(disc_y_loss, discriminator_y.trainable_variables),
            discriminator_y.trainable_variables))
    return total_gen_g, total_gen_f, disc_x_loss, disc_y_loss, ssim_l

In [ ]:
# Cell 13 — Instantiate Models
generator_g     = build_generator()
generator_f     = build_generator()
discriminator_x = build_discriminator()
discriminator_y = build_discriminator()

In [ ]:
# Cell 14 — Shape check
x_test = tf.random.normal((4, 256, 256, 1))
print('Generator G output:    ', generator_g(x_test).shape)
print('Generator F output:    ', generator_f(x_test).shape)
print('Discriminator X output:', discriminator_x(x_test).shape)
print(f'Generator params:       {generator_g.count_params():,}')
print(f'Discriminator params:   {discriminator_x.count_params():,}')

In [ ]:
# Cell 15 — Subject-Level Train/Validation Split and Main Training Loop
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd

CSV_LOG = os.path.join(OUTPUT_DIR, 'metrics.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(CSV_LOG, 'w') as f:
    f.write('epoch,train_ssim,train_psnr,val_ssim,val_psnr,ssim_loss\n')

# ---------------------------------------------------------------
# Split at SUBJECT level using an 85:15 train/validation ratio.
# All slices from the same subject remain in one partition.
# ---------------------------------------------------------------
subject_df = (
    pairs_df[['subject_id']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# No class labels are used here because this CycleGAN domain contains
# only the supplied T1/T2 domain images. The important requirement is
# subject-level separation.
train_subjects, val_subjects = train_test_split(
    subject_df['subject_id'].values,
    test_size=0.15,
    random_state=SEED
)

train_subjects = set(train_subjects)
val_subjects = set(val_subjects)

assert train_subjects.isdisjoint(val_subjects)

train_pairs = pairs_df[pairs_df['subject_id'].isin(train_subjects)].reset_index(drop=True)
val_pairs   = pairs_df[pairs_df['subject_id'].isin(val_subjects)].reset_index(drop=True)

def load_pair_arrays(pair_df):
    t1_list, t2_list = [], []

    for _, row in pair_df.iterrows():
        try:
            t1 = load_img(
                row['t1_path'],
                color_mode='grayscale',
                target_size=(IMG_HEIGHT, IMG_WIDTH)
            )
            t2 = load_img(
                row['t2_path'],
                color_mode='grayscale',
                target_size=(IMG_HEIGHT, IMG_WIDTH)
            )

            t1_list.append(img_to_array(t1) / 127.5 - 1.0)
            t2_list.append(img_to_array(t2) / 127.5 - 1.0)

        except Exception as e:
            print(f"Skipping {row['slice_key']}: {e}")

    return np.asarray(t1_list, dtype=np.float32), np.asarray(t2_list, dtype=np.float32)

t1_train, t2_train = load_pair_arrays(train_pairs)
t1_val, t2_val     = load_pair_arrays(val_pairs)

print(f'Train subjects: {len(train_subjects)} | slices: {len(t1_train)}')
print(f'Val subjects:   {len(val_subjects)} | slices: {len(t1_val)}')

n_train = len(t1_train)
n_val   = len(t1_val)

train_steps = n_train // BATCH_SIZE
val_steps   = n_val // BATCH_SIZE

if train_steps == 0 or val_steps == 0:
    raise ValueError(
        'Not enough images for the selected BATCH_SIZE. '
        'Reduce BATCH_SIZE or check the dataset.'
    )

def metrics_on_batch(imgs, reals):
    fake = generator_g(imgs, training=False).numpy()
    reals = reals.numpy() if hasattr(reals, 'numpy') else reals

    ss, ps = [], []
    for r, fi in zip(reals, fake):
        s, p = evaluate_ssim_psnr(r.squeeze(), fi.squeeze())
        ss.append(s)
        ps.append(p)

    return np.mean(ss), np.mean(ps)

for epoch in range(1, EPOCHS + 1):
    print(f'\nEpoch {epoch}/{EPOCHS}')
    train_ss, train_ps, train_sl = [], [], []

    for step in tqdm(range(train_steps), desc='Train'):
        idx = step * BATCH_SIZE

        t1_b = tf.cast(
            t1_train[idx:idx+BATCH_SIZE], tf.float32
        )
        t2_b = tf.cast(
            t2_train[idx:idx+BATCH_SIZE], tf.float32
        )

        _, _, _, _, sl = train_step(
            t1_b, t2_b,
            generator_g, generator_f,
            discriminator_x, discriminator_y
        )

        train_sl.append(float(sl))

        if step % 10 == 0:
            ss, ps = metrics_on_batch(t1_b, t2_b)
            train_ss.append(ss)
            train_ps.append(ps)

    val_ss, val_ps = [], []

    for step in tqdm(range(val_steps), desc='Val  '):
        idx = step * BATCH_SIZE

        t1_b = tf.cast(
            t1_val[idx:idx+BATCH_SIZE], tf.float32
        )
        t2_b = tf.cast(
            t2_val[idx:idx+BATCH_SIZE], tf.float32
        )

        ss, ps = metrics_on_batch(t1_b, t2_b)
        val_ss.append(ss)
        val_ps.append(ps)

    tr_ss, tr_ps = np.mean(train_ss), np.mean(train_ps)
    vl_ss, vl_ps = np.mean(val_ss), np.mean(val_ps)
    mn_sl = np.mean(train_sl)

    print(
        f'Train  SSIM: {tr_ss:.4f}  '
        f'PSNR: {tr_ps:.2f} dB  '
        f'SSIM-loss: {mn_sl:.4f}'
    )
    print(
        f'Val    SSIM: {vl_ss:.4f}  '
        f'PSNR: {vl_ps:.2f} dB'
    )

    with open(CSV_LOG, 'a') as f:
        f.write(
            f'{epoch},{tr_ss},{tr_ps},{vl_ss},{vl_ps},{mn_sl}\n'
        )

print('Training complete - metrics saved to', CSV_LOG)


In [ ]:
# Cell 16 — Results Grid (unchanged from baseline)
import matplotlib.pyplot as plt
n_show = min(5, len(t1_val))
fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
for col in range(n_show):
    t1      = t1_val[col]
    t2_real = t2_val[col]
    t2_fake = generator_g(np.expand_dims(t1, 0), training=False)[0].numpy()
    axes[0, col].imshow(t1.squeeze(),      cmap='gray'); axes[0, col].set_title('T1 input')
    axes[1, col].imshow(t2_fake.squeeze(), cmap='gray'); axes[1, col].set_title('Generated T2')
    axes[2, col].imshow(t2_real.squeeze(), cmap='gray'); axes[2, col].set_title('Real T2')
    for ax in axes[:, col]: ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'final_grid.png'), dpi=300)
plt.show()

In [ ]:
# Cell 17 — Training Curves (SSIM training-loss panel added)
import pandas as pd
df = pd.read_csv(CSV_LOG)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4), sharex=True)

ax1.plot(df.epoch, df.train_ssim, label='Train SSIM', marker='o', markevery=5)
ax1.plot(df.epoch, df.val_ssim,   label='Val SSIM',   marker='s', markevery=5)
ax1.set_ylabel('SSIM'); ax1.legend(); ax1.grid(alpha=.3); ax1.set_ylim(0, 1)

ax2.plot(df.epoch, df.train_psnr, label='Train PSNR', marker='o', markevery=5)
ax2.plot(df.epoch, df.val_psnr,   label='Val PSNR',   marker='s', markevery=5)
ax2.set_ylabel('PSNR (dB)'); ax2.legend(); ax2.grid(alpha=.3)
ax2.set_xlabel('Epoch')

ax3.plot(df.epoch, df.ssim_loss, color='red', label='SSIM loss (1-SSIM)')
ax3.set_ylabel('1 - SSIM'); ax3.legend(); ax3.grid(alpha=.3)
ax3.set_title('SSIM Training Loss (novel objective)')

gap_ssim = np.abs(df.train_ssim - df.val_ssim)
print(f'Max generalisation gap (SSIM): {gap_ssim.max():.4f}')
best_ssim = df.val_ssim.max()
thresh = 0.9 * best_ssim
epoch_90 = df[df.val_ssim >= thresh].epoch.min()
print(f'90% of best Val-SSIM reached at epoch {epoch_90}  (best={best_ssim:.3f})')

plt.suptitle('CycleGAN SW-MSA + Freq-Discriminator Training Curves')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eval_curves_ssim_psnr.pdf'))
plt.savefig(os.path.join(OUTPUT_DIR, 'eval_curves_ssim_psnr.png'), dpi=300)
plt.show()

In [ ]:
# Cell 18 — Save weights and artefacts
# Change PERMANENT_DIR to your preferred local or mounted output location.
PERMANENT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(PERMANENT_DIR, exist_ok=True)

generator_g.save_weights(
    os.path.join(PERMANENT_DIR, 'gen_g.weights.h5')
)
generator_f.save_weights(
    os.path.join(PERMANENT_DIR, 'gen_f.weights.h5')
)
discriminator_x.save_weights(
    os.path.join(PERMANENT_DIR, 'disc_x.weights.h5')
)
discriminator_y.save_weights(
    os.path.join(PERMANENT_DIR, 'disc_y.weights.h5')
)

print('All model weights saved to:', PERMANENT_DIR)
print('Metrics saved to:', CSV_LOG)


In [ ]:
# Cell 19 — Reproducible Evaluation Sheet (unchanged from baseline)
np.random.seed(42); tf.random.set_seed(42)
n_show = 5
fig, axes = plt.subplots(n_show, 6, figsize=(18, 3 * n_show))
fig.patch.set_facecolor('white')
for row in range(n_show):
    t1_real   = t1_val[row]
    t2_real   = t2_val[row]
    t2_fake   = generator_g(np.expand_dims(t1_real, 0), training=False)[0].numpy()
    t1_cycled = generator_f(np.expand_dims(t2_fake,  0), training=False)[0].numpy()
    images = [t1_real, t2_fake, t2_real, t1_cycled, t1_real, np.abs(t1_real - t1_cycled)]
    titles = ['T1 input', 'Generated T2', 'Real T2', 'Cycled T1', 'Real T1', '|Delta T1|']
    for col, (img, title) in enumerate(zip(images, titles)):
        img = np.asarray(img)
        while img.ndim > 2: img = img.squeeze(axis=-1)
        if img.ndim != 2: img = img[..., 0]
        axes[row, col].imshow(img, cmap='gray', vmin=-1, vmax=1)
        axes[row, col].set_title(title, fontsize=10)
        axes[row, col].axis('off')
plt.suptitle('CycleGAN SW-MSA  T1 <-> T2  Evaluation Sheet', fontsize=16)
plt.tight_layout()
out_png = os.path.join(OUTPUT_DIR, 'eval_sheet.png')
out_pdf = os.path.join(OUTPUT_DIR, 'eval_sheet.pdf')
plt.savefig(out_png, dpi=300, bbox_inches='tight')
plt.savefig(out_pdf, bbox_inches='tight')
plt.show()
print('Evaluation sheet saved ->', out_png, '&', out_pdf)

## Reproducibility notes
- The MRI dataset is not included in the repository.
- T1/T2 slices are matched by slice key, while train/validation splitting is performed at the subject level.
- All slices belonging to one subject remain in a single partition.
- Set `DATA_ROOT` before running the notebook.
- The original CycleGAN objective and model components are retained; only the data partitioning and path handling were revised for reproducibility and leakage prevention.
